# Query Relationships - GraphRAG

This notebook demonstrates how to query and analyze the `relationships.parquet` file from GraphRAG by source and target entities.

## 1. Import Required Libraries

Import necessary libraries for working with Parquet files.

In [ ]:
# Import Required Libraries
import pandas as pd
import pyarrow.parquet as pq
import os
from IPython.display import display, Markdown

# Set display options for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.width', None)

# Set the path to the relationships parquet file
parquet_path = '/home/chuaxu/projects/graphrag/ragtest/output/relationships.parquet'

# Verify the file exists
if os.path.exists(parquet_path):
    print("✅ Relationships parquet file found")
    print(f"📁 Path: {parquet_path}")
else:
    print(f"❌ File not found: {parquet_path}")

## 2. Load Relationships Parquet File

Load the relationships.parquet file into a DataFrame using pandas.

In [ ]:
# Load the parquet file into a DataFrame
df = pd.read_parquet(parquet_path)

print("="*80)
print("✅ SUCCESSFULLY LOADED RELATIONSHIPS PARQUET FILE")
print("="*80)
print(f"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("="*80)

## 3. Explore Data Structure

Examine the DataFrame structure, including column names, data types, and sample data.

In [ ]:
# Display column information
print("="*80)
print("📋 COLUMNS & DATA TYPES")
print("="*80)
for idx, (col, dtype) in enumerate(df.dtypes.items(), 1):
    print(f"{idx:2d}. {col:20s} → {dtype}")
print("="*80)

In [ ]:
# Display first few rows
print("="*80)
print("🔍 FIRST 5 ROWS")
print("="*80)
display(df.head())

In [ ]:
# Display summary statistics for weight and combined_degree
print("="*80)
print("📊 SUMMARY STATISTICS (Weight & Combined Degree)")
print("="*80)
display(df[['weight', 'combined_degree']].describe())

## 4. Configuration: Set Source and Target Entities

Configure the entities you want to search for in relationships.

In [ ]:
# ⚙️ CONFIGURATION: Set the entities you want to search for
# Change these variables to search for different entity relationships
SEARCH_SOURCE = 'JACOB MARLEY'       # Source entity (leave empty '' to search all sources)
SEARCH_TARGET = 'GHOST'  # Target entity (leave empty '' to search all targets)

print("="*80)
print(f"🔧 SEARCH CONFIGURATION")
print("="*80)
print(f"📝 Source Entity: '{SEARCH_SOURCE if SEARCH_SOURCE else 'ANY'}'")
print(f"📝 Target Entity: '{SEARCH_TARGET if SEARCH_TARGET else 'ANY'}'")
print("="*80)

## 5. Query Relationships

Search for relationships by source and target entities.

In [ ]:
# Method 1: Query by exact source and target match
print("="*80)
print(f"🔍 QUERY 1: Exact Match (Source='{SEARCH_SOURCE}', Target='{SEARCH_TARGET}')")
print("="*80)

result = df[(df['source'] == SEARCH_SOURCE) & (df['target'] == SEARCH_TARGET)]
print(f"✅ Found {len(result)} relationship(s)\n")

if len(result) > 0:
    display(result[['human_readable_id', 'source', 'target', 'weight', 'combined_degree', 'description']])
    
    # Display full description
    print("\n" + "="*80)
    print("📝 FULL DESCRIPTION")
    print("="*80)
    for idx, desc in enumerate(result['description'].values, 1):
        print(f"\nDescription {idx}:")
        print("-"*80)
        print(desc)
        print("-"*80)
else:
    print(f"❌ No exact match found")
    print("💡 Try the case-insensitive search below!")
    
print("="*80)

In [ ]:
# Method 2: Query by source only (case-insensitive)
print("="*80)
print(f"🔍 QUERY 2: By Source Entity (case-insensitive: '{SEARCH_SOURCE}')")
print("="*80)

if SEARCH_SOURCE:
    result_by_source = df[df['source'].str.contains(SEARCH_SOURCE, case=False, na=False)]
    print(f"✅ Found {len(result_by_source)} relationship(s)\n")
    
    if len(result_by_source) > 0:
        # Sort by weight descending
        result_by_source_sorted = result_by_source.sort_values('weight', ascending=False)
        display(result_by_source_sorted[['source', 'target', 'weight', 'combined_degree', 'description']].head(20))
        
        # Show statistics
        print("\n" + "="*80)
        print("📊 STATISTICS")
        print("="*80)
        print(f"Total relationships: {len(result_by_source):,}")
        print(f"Average weight: {result_by_source['weight'].mean():.2f}")
        print(f"Total weight: {result_by_source['weight'].sum():.2f}")
        print(f"Max weight: {result_by_source['weight'].max():.2f}")
        print(f"Unique targets: {result_by_source['target'].nunique():,}")
    else:
        print("❌ No matches found")
else:
    print("⚠️ SEARCH_SOURCE is empty. Set a source entity to search.")
    
print("="*80)

In [ ]:
# Method 3: Query by target only (case-insensitive)
print("="*80)
print(f"🔍 QUERY 3: By Target Entity (case-insensitive: '{SEARCH_TARGET}')")
print("="*80)

if SEARCH_TARGET:
    result_by_target = df[df['target'].str.contains(SEARCH_TARGET, case=False, na=False)]
    print(f"✅ Found {len(result_by_target)} relationship(s)\n")
    
    if len(result_by_target) > 0:
        # Sort by weight descending
        result_by_target_sorted = result_by_target.sort_values('weight', ascending=False)
        display(result_by_target_sorted[['source', 'target', 'weight', 'combined_degree', 'description']].head(20))
        
        # Show statistics
        print("\n" + "="*80)
        print("📊 STATISTICS")
        print("="*80)
        print(f"Total relationships: {len(result_by_target):,}")
        print(f"Average weight: {result_by_target['weight'].mean():.2f}")
        print(f"Total weight: {result_by_target['weight'].sum():.2f}")
        print(f"Max weight: {result_by_target['weight'].max():.2f}")
        print(f"Unique sources: {result_by_target['source'].nunique():,}")
    else:
        print("❌ No matches found")
else:
    print("⚠️ SEARCH_TARGET is empty. Set a target entity to search.")
    
print("="*80)

In [ ]:
# Method 4: Query by either source OR target (bidirectional search)
SEARCH_ENTITY = 'SCROOGE'  # Entity that could be either source or target

print("="*80)
print(f"🔍 QUERY 4: Bidirectional Search (Entity: '{SEARCH_ENTITY}')")
print("="*80)

if SEARCH_ENTITY:
    # Find relationships where entity is either source or target
    result_bidirectional = df[
        (df['source'].str.contains(SEARCH_ENTITY, case=False, na=False)) |
        (df['target'].str.contains(SEARCH_ENTITY, case=False, na=False))
    ]
    print(f"✅ Found {len(result_bidirectional)} relationship(s)\n")
    
    if len(result_bidirectional) > 0:
        # Sort by weight descending
        result_bidirectional_sorted = result_bidirectional.sort_values('weight', ascending=False)
        display(result_bidirectional_sorted[['source', 'target', 'weight', 'combined_degree']].head(20))
        
        # Show statistics
        print("\n" + "="*80)
        print("📊 STATISTICS")
        print("="*80)
        print(f"Total relationships: {len(result_bidirectional):,}")
        print(f"As source: {len(result_bidirectional[result_bidirectional['source'].str.contains(SEARCH_ENTITY, case=False, na=False)]):,}")
        print(f"As target: {len(result_bidirectional[result_bidirectional['target'].str.contains(SEARCH_ENTITY, case=False, na=False)]):,}")
        print(f"Average weight: {result_bidirectional['weight'].mean():.2f}")
        print(f"Total weight: {result_bidirectional['weight'].sum():.2f}")
        print(f"Connected entities: {len(set(result_bidirectional['source']) | set(result_bidirectional['target'])):,}")
    else:
        print("❌ No matches found")
else:
    print("⚠️ SEARCH_ENTITY is empty. Set an entity to search.")
    
print("="*80)

## 6. Top Relationships Analysis

Analyze the strongest relationships by weight and combined degree.

In [ ]:
# Top relationships by weight
print("="*80)
print("📊 TOP 10 RELATIONSHIPS BY WEIGHT")
print("="*80)

top_by_weight = df.nlargest(10, 'weight')[['source', 'target', 'weight', 'combined_degree', 'description']]
display(top_by_weight)
print("="*80)

In [ ]:
# Top relationships by combined_degree
print("="*80)
print("📊 TOP 10 RELATIONSHIPS BY COMBINED DEGREE")
print("="*80)

top_by_degree = df.nlargest(10, 'combined_degree')[['source', 'target', 'weight', 'combined_degree', 'description']]
display(top_by_degree)
print("="*80)

## 7. Entity Network Analysis

Analyze the network of entities and their connections.

In [ ]:
# Find the most connected entities
print("="*80)
print("🕸️ MOST CONNECTED ENTITIES")
print("="*80)

# Count outgoing connections (as source)
source_counts = df['source'].value_counts()
# Count incoming connections (as target)
target_counts = df['target'].value_counts()
# Total connections
all_entities = list(set(df['source']) | set(df['target']))
entity_connections = {}
for entity in all_entities:
    outgoing = source_counts.get(entity, 0)
    incoming = target_counts.get(entity, 0)
    entity_connections[entity] = outgoing + incoming

# Sort by total connections
sorted_entities = sorted(entity_connections.items(), key=lambda x: x[1], reverse=True)

print(f"\n{'Entity':<40} {'Total Connections':>20} {'As Source':>15} {'As Target':>15}")
print("-"*95)
for entity, total in sorted_entities[:15]:
    outgoing = source_counts.get(entity, 0)
    incoming = target_counts.get(entity, 0)
    print(f"{entity:<40} {total:>20,} {outgoing:>15,} {incoming:>15,}")
print("="*80)

## 8. Export Query Results

Export the query results to various formats.

In [ ]:
# Export results to files
EXPORT_ENTITY = 'SCROOGE'  # Change this to export results for a specific entity

print("="*80)
print("💾 EXPORT RESULTS")
print("="*80)

if EXPORT_ENTITY:
    # Get all relationships for this entity
    export_data = df[
        (df['source'].str.contains(EXPORT_ENTITY, case=False, na=False)) |
        (df['target'].str.contains(EXPORT_ENTITY, case=False, na=False))
    ]
    
    if len(export_data) > 0:
        # Export to CSV
        csv_path = f'/home/chuaxu/projects/graphrag/{EXPORT_ENTITY.lower()}_relationships.csv'
        export_data.to_csv(csv_path, index=False)
        print(f"✅ Exported to CSV: {csv_path}")
        
        # Export to JSON
        json_path = f'/home/chuaxu/projects/graphrag/{EXPORT_ENTITY.lower()}_relationships.json'
        export_data.to_json(json_path, orient='records', indent=2)
        print(f"✅ Exported to JSON: {json_path}")
        
        # Export to Parquet
        parquet_output = f'/home/chuaxu/projects/graphrag/{EXPORT_ENTITY.lower()}_relationships.parquet'
        export_data.to_parquet(parquet_output, index=False)
        print(f"✅ Exported to Parquet: {parquet_output}")
        
        print(f"\n📊 Exported {len(export_data)} relationship(s)")
    else:
        print(f"❌ No relationships found for '{EXPORT_ENTITY}'")
else:
    print("⚠️ EXPORT_ENTITY is empty. Set an entity name to export.")
    
print("="*80)